# Model sharing

## What does it mean to share a model?

Model sharing can mean multiple things:

```mermaid
mindmap
    root((Shared model))
        Model weights
            Specifies model's parameters
        Model code
            Specifies how the model can be run
        Model card
            Describes the model to users
        Training code
            Specifies how the model can be trained
        Training data
            The data used to train the model
```

Let's look at these in detail:

### Model weights

Model weights is the simplest to understand: it is just the parameter weights of the model.

These weights are often [serialized versions](https://en.wikipedia.org/wiki/Serialization) of the numbers stored in training framework's memory. While doing training these weights are often stored by pickling Python objects ([PyTorch's serialization routines](https://docs.pytorch.org/docs/stable/notes/serialization.html) work this way), but because Python's [pickle](https://docs.python.org/3/library/pickle.html)-module is not secure, they are rarely stored in these formats when shared.

Because we do not want to execute random code provided to us by strangers over the internet, multiple different formats have been designed to fix this problem.

[ONNX](https://onnx.ai/) is a open standard for sharing machine learning models and [PyTorch supports is natively](https://docs.pytorch.org/tutorials/beginner/onnx/export_simple_model_to_onnx_tutorial.html). It is widely used, especially in industry, where trained models do inference on various hardware devices.  

ONXX converts the whole computation graph of the model into operations that can then be stored in the serialization format. Same is done for the parameters.

Another popular format, especially among scientists and ML designers, is [safetensors](https://huggingface.co/docs/safetensors). This format was created by Hugging Face and it is used in many repos in Hugging Face Hub. Safetensors focuses on serializing the weights, so getting a working model from safetensors file requires access to the module structure where there parameters will be placed.

#### Sharded model weight files on the Hub

For large models, you might see multiple files like `model-00001-of-0000N.safetensors` instead of a single `model.safetensors`. This is called **sharding**: the weights are split across multiple files, and an index file (typically `model.safetensors.index.json`) maps each parameter name to the shard file that contains it.

This is usually done **automatically at save time** by the tooling that writes the checkpoint (for example, Hugging Face Transformers), based on a maximum shard size. The Hub typically just stores whatever files you upload; it does not take one big checkpoint and automatically split it into shards after upload.

For example, in Transformers you can control the behavior with `max_shard_size`:

```python
model.save_pretrained("out_dir", safe_serialization=True, max_shard_size="5GB")
# and then push the resulting directory to the Hub
```

### Model code

Like mentioned in the previous section, sometimes getting access to the model specification is needed to construct the model.

Sometimes the code is given as code in e.g. Github repositories, but sometimes the configuration is given as specification.

For example, Hugging Face uses a concept called [AutoModel](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModel) that picks a model from an pre-existing list of model specifications. These models are then initialized based on [AutoConfig](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoConfig) given in model's repository. So when you're calling `AutoModelForCausalLM.from_pretrained`, the following happens:

```mermaid
flowchart TD
    R[User asks for model] --> F["AutoModelForCausalLM.from_pretrained(...)"];
    F --> H[Hugging Face Hub checks the repository];
    H --> C[Repository contains an AutoConfig];
    C --> M[AutoModel is initialized with layer configuration from AutoConfig];
    M --> R
```

A good example of this is [gpt-oss-120b's model configuration](https://huggingface.co/openai/gpt-oss-120b/blob/main/config.json).
It utilizes [GptOssForCausalLM](https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt_oss/modeling_gpt_oss.py#L637), which in turn is a subclass is [GptOssPreTrainedModel](https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt_oss/modeling_gpt_oss.py#L422), which in turn is a subclass of [PreTrainedModel](https://github.com/huggingface/transformers/blob/main/src/transformers/modeling_utils.py#L1644).

The complex model structure can be represented by a simple json-file for all GPT-like models.

### Model card

The idea of a machine learning model card was introduced in 2018 by [Mitchell et al.](https://arxiv.org/abs/1810.03993). The basic idea of a model card is that it should contain information on the intented use cases of the model and possible limitations of the model.

Since then, the concept has been used by all of the major players in the field: [Hugging Face](https://huggingface.co/docs/hub/model-cards), [OpenAI](https://platform.openai.com/docs/models/system-cards/), [Google](https://modelcards.withgoogle.com/), [Meta](https://www.llama.com/docs/model-cards-and-prompt-formats/) to name a few.

It is good to remember that writing a model card is an important piece of sharing a model.

### Training code

Compared to the others mentioned before, training code is not shared as often. Especially in the field of huge models like foundation large language models training can be so expensive, that keeping the training code hidden can provide a major competitive advantage to AI developers.

In purely scientific fields sharing the training code is much more common and GitHub is the most common way of sharing code. A strong, shareable code base goes beyond the core training scripts: adding clear documentation, dependency specifications, configuration files, dataset preparation instructions, and (when possible) pretrained checkpoints makes it much easier for others (and future you) to rerun results, learn from your work, and build on it.

### Training strategy

It also helps a lot to describe the full training strategy, since modern machine learning systems are often trained in multiple stages rather than a single straightforward step. Many models use multi-stage pipelines—for example pretraining on large generic datasets followed by fine-tuning on task-specific data. Others may mix learning paradigms such as supervised learning, unsupervised or self-supervised learning, reinforcement learning, or human feedback–based alignment methods. When that’s the case, spelling out the sequence of stages, the data used at each step, and how objectives or hyperparameters evolve across phases (e.g., curricula, filtering/mixing, schedules, post-training or alignment steps) gives readers the context they need to reproduce the behavior and understand what’s driving performance.

### Training data

Sharing training data is another complicated topic. Similar to training codes, possessing more and better quality training data will provide companies with competitive advantages and thus many of the training datasets are not shared. Questions of licensing and data ownership also limit the possibility of sharing the training data.

[Hugging Face datasets](https://huggingface.co/datasets) provides lots of datasets that are commonly used for various tasks.

## Where models are shared

Models are nowadays shared through various sites, but [Hugging Face Hub](https://huggingface.co/models) is one of the most popular places for [model sharing](https://huggingface.co/docs/transformers/model_sharing).

[Zenodo](https://zenodo.org/) and other similar publicly funded storage solutions also contain datasets, but they often lose in ease of use to Hugging Face.

### Hub model repos are git-like

A Hub **model repository** works much like a normal **git repository**:

| Concept | On the Hub |
|--------|------------|
| Repository | Identified by `REPO_ID` (e.g. `username/my-model`) |
| Files | Weights, config, tokenizer, README, … |
| History | Every upload creates a **commit** you can browse |
| Versions | **Branches** and **tags** (called *revisions*) point at specific commits |

---

### Storing large files: Git LFS vs Xet

Weight files are too large for plain git blobs. Hugging Face has moved from **Git LFS** to **[Xet](https://huggingface.co/docs/hub/xet/index)** for many uploads.

| | **Git LFS** (older) | **Xet** (newer) |
|---|---------------------|-----------------|
| Idea | Store whole large files as LFS objects | Split files into **content-addressed chunks** |
| Storage | LFS pointers in git; **whole files** in S3 (SHA256 keys) | Same pointer pattern in git; **chunks** in S3 / CAS |
| Transfer | Often re-upload/download entire file | Client sends/receives **only changed chunks** |
| Deduplication | File-level (same hash → same object) | Chunk-level across **commits** and **artifacts** |

Both backends use **S3** for the actual bytes; Xet changes *how* files are split, deduplicated, and transferred—not where blobs live.

Xet tends to mean **faster downloads** when you pull the same weights again or when chunks overlap with another model.

---

### How to interact with a repo

Because the layout is git-like, you *can* use ordinary **Git** (clone, pull, push) or the **[Hugging Face CLI](https://huggingface.co/docs/huggingface_hub/guides/cli)** (`hf upload`, `hf download`, …) when you want file-level control.

In this notebook—and in most Transformers workflows—you interact with a Hub repo through the library instead: **`from_pretrained(REPO_ID)`** to download and **`push_to_hub(REPO_ID, …)`** to upload. Those methods handle authentication, sharding, and large-file transfer (via Xet or LFS under the hood) so you do not need to set up git + LFS yourself. The example below uses `model.push_to_hub`, `tokenizer.push_to_hub`, and `ModelCard.push_to_hub`.

## Example: push a checkpoint to Hugging Face Hub Model Repository

Share a model via Transformers is the most common approach. Many classes (`models`, `tokenizers`) have a built-in [push_to_hub() method](https://huggingface.co/docs/transformers/model_sharing#pushtohubmixin).

The cells below show how to upload a local Hugging Face `Trainer` checkpoint (for example from the IMDB fine-tuning example, `pytorch_imdb_gpt.py`) to a new or existing [model repository on the Hub](https://huggingface.co/docs/hub/models-the-hub). Set `CHECKPOINT_DIR`, `REPO_ID`, and `HF_TOKEN` via environment variables or by editing the parameter cell. A [personal access token](https://huggingface.co/docs/hub/security-tokens) with write access is required to create or update the remote repository.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("secrets/.env", override=False)

In [ ]:
# --- Parameters (edit these, or set via environment in secrets/.env) ---
token = os.getenv("HF_TOKEN")
CHECKPOINT_DIR = os.getenv("CHECKPOINT_DIR", "")
REPO_ID = os.getenv("REPO_ID", "")
BASE_MODEL = os.getenv("BASE_MODEL", "EleutherAI/gpt-neo-125m")
HF_PRIVATE = os.getenv("HF_PRIVATE", "false").strip().lower() in ("1", "true", "yes", "y")

assert CHECKPOINT_DIR, "Set CHECKPOINT_DIR (env var or edit this cell)."
assert REPO_ID, "Set REPO_ID (env var or edit this cell)."

ckpt = Path(CHECKPOINT_DIR).expanduser().resolve()
print(f"Checkpoint: {ckpt}")
print(f"Repo: {REPO_ID} (private={HF_PRIVATE})")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import ModelCard, ModelCardData

if not ckpt.exists():
    raise SystemExit(f"Checkpoint dir does not exist: {ckpt}")

model = AutoModelForCausalLM.from_pretrained(str(ckpt))

# Some Trainer checkpoints may not include tokenizer files; fall back to base model.
try:
    tokenizer = AutoTokenizer.from_pretrained(str(ckpt), use_fast=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token



In [ ]:
MAX_SHARD_SIZE = "30GB"
model.push_to_hub(REPO_ID, private=HF_PRIVATE, token=token, max_shard_size=MAX_SHARD_SIZE)
tokenizer.push_to_hub(REPO_ID, private=HF_PRIVATE, token=token)

In [ ]:
# Add a model card (a README.md file will be created if it doesn't exist) to the repository
card_data = ModelCardData(
    language="en",
    license="apache-2.0",
    library_name="transformers",
    tags=["pytorch", "causal-lm", "imdb", "text-generation"],
    base_model=BASE_MODEL,
    datasets=["stanfordnlp/imdb"],
)
description = (
    f"This checkpoint was fine-tuned for **sentiment-conditioned text generation** on the IMDB "
    f"reviews dataset, starting from `{BASE_MODEL}`.\n\n"
    f"**Local source checkpoint:** `{ckpt}`\n\n"
    "Edit this card on the Hub or in this notebook to add evaluation numbers, limitations, and intended use."
)
card = ModelCard.from_template(card_data, model_description=description)
card.push_to_hub(
    REPO_ID,
    token=token,
    commit_message="Add model card (README.md)",
)

print(f"Pushed checkpoint to Hub: {REPO_ID}")
print(f"Source checkpoint: {ckpt}")
print(f"max_shard_size: {MAX_SHARD_SIZE}")

## Summary: ways to interact with the Hugging Face Hub

Hub **model repos** are git-like (files, commits, branches/tags). You can work with them at several levels—from clicking in the browser to calling Python APIs. Pick the layer that matches your task.

| Method | What you use | Best for | Typical operations |
|--------|----------------|----------|-------------------|
| **Hub UI** | [huggingface.co](https://huggingface.co) in a browser | Browsing, light edits, collaboration | Create repo, upload files (drag-and-drop), edit `README.md` / model card, set visibility, view commit history, discuss in community tabs |
| **Git (+ LFS / Xet)** | `git`, [Git LFS](https://git-lfs.com/), Hub’s [Xet](https://huggingface.co/docs/hub/xet/index) backend | Full version control, mirroring a repo locally | `git clone https://huggingface.co/USER/REPO`, edit files, `git commit`, `git push` (large weights go through LFS/Xet, not plain git blobs) |
| **`huggingface_hub` + `hf` CLI** | Python package [`huggingface_hub`](https://huggingface.co/docs/huggingface_hub); CLI [`hf`](https://huggingface.co/docs/huggingface_hub/guides/cli) | File-level upload/download without Transformers | `hf download REPO_ID --local-dir ./out`, `hf upload REPO_ID ./folder`, `HfApi().create_repo(...)`, `snapshot_download`, `upload_folder` |
| **Transformers (and similar libs)** | [`transformers`](https://huggingface.co/docs/transformers/model_sharing) (+ Datasets, etc.) | ML workflows: load/save models with correct config & sharding | `AutoModel.from_pretrained(REPO_ID)`, `model.push_to_hub(REPO_ID)`, `Trainer` + `hub_model_id`, `ModelCard.push_to_hub` |